In [137]:
%%bash
ep="episode_000003"
dataset_dir="lerobot_data/TEST_DATASET"

ls "${dataset_dir}/data/chunk-000/${ep}.parquet"

rm -f "${dataset_dir}/data/chunk-000/${ep}.parquet"
rm -f "${dataset_dir}/videos/chunk-000/observation.images.rgb_left/${ep}.mp4"
rm -f "${dataset_dir}/videos/chunk-000/observation.images.rgb_right/${ep}.mp4"
rm -f "${dataset_dir}/videos/chunk-000/observation.images.rgb_static/${ep}.mp4"
rm -f "${dataset_dir}/data/chunk-000/${ep}.parquet"


ls: cannot access 'lerobot_data/TEST_DATASET/data/chunk-000/episode_000003.parquet': No such file or directory
 directory


In [154]:
import os
import json
from pathlib import Path

dataset_root_dir = Path("lerobot_data/TEST_DATASET")
meta_dir = dataset_root_dir / "meta"
meta_dir.mkdir(exist_ok=True)  # make sure meta exists

blacklist_path = meta_dir / "blacklist.json"
episodes_path = meta_dir / "episodes.jsonl"

# Array of episode indices you want to remove
ids_remove = [3, 5, 7]  # store as ints for consistent comparisons

# Load existing blacklist or create an empty one
if blacklist_path.exists():
    with open(blacklist_path, "r") as f:
        try:
            blacklisted_indices = set(json.load(f))
        except json.JSONDecodeError:
            blacklisted_indices = set()
else:
    print(f"No blacklist found at {blacklist_path}, creating new one.")
    blacklisted_indices = set()

# Add all episode indices to blacklist
blacklisted_indices.update(ids_remove)

# Save back to disk
with open(blacklist_path, "w") as f_out:
    json.dump(sorted(blacklisted_indices), f_out, indent=2)

# (Optional) Verify that episodes exist before/after blacklisting
if episodes_path.exists():
    with open(episodes_path, "r") as f:
        episodes = [json.loads(line) for line in f]
        episode_ids = {ep["episode_index"] for ep in episodes}

    missing = [ep for ep in ids_remove if ep not in episode_ids]
    if missing:
        print(f"⚠️ Warning: The following episodes were not found in episodes.jsonl: {missing}")

print(f"✅ Successfully added {len(ids_remove)} episode(s) to blacklist.")
